# 03 — Cross-Calibration

Calibrate Ultra460, Ultra321, and Pico017 against the Picarro (CH4) and certified gas standards (C3H8) using the Feb 12 calibration sequence.
Validate on Feb 6.  C2H6 is cross-calibrated across all WYO co-deployment days using Ultra460 as the reference.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from pathlib import Path

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})


In [ ]:
MERGED_DIR = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026/merged')

# Certified concentrations (calibration_details.txt)
CERT = {
    'N2_zero':   {'CH4_ppm': 0.0,    'C3H8_ppm': 0.0,  'C2H6_ppb': 0.0},
    'NOAA':      {'CH4_ppm': 2.0012, 'C3H8_ppm': None, 'C2H6_ppb': 1.63},
    'Dilution1': {'CH4_ppm': 5.736,  'C3H8_ppm': 1.0,  'C2H6_ppb': None},
    'Dilution2': {'CH4_ppm': 11.471, 'C3H8_ppm': 2.0,  'C2H6_ppb': None},
    'Dilution3': {'CH4_ppm': 17.2,   'C3H8_ppm': 3.0,  'C2H6_ppb': None},
    'Dilution4': {'CH4_ppm': 28.878, 'C3H8_ppm': 5.0,  'C2H6_ppb': None},
    'Dilution5': {'CH4_ppm': 57.333, 'C3H8_ppm': 10.0, 'C2H6_ppb': None},
}

COLORS = {
    'Picarro':  '#1f77b4',
    'Ultra460': '#ff7f0e',
    'Ultra321': '#2ca02c',
    'Pico017':  '#d62728',
}


In [ ]:
# Stable calibration windows (UTC) — precise ranges from calibration_details.txt.
# Feb 12 is the primary calibration; Feb 6 is validation only.

FEB12_WINDOWS = [
    {'label': 'N2_zero',   'cert_key': 'N2_zero',
     'stable_start': '2026-02-12 22:25:30', 'stable_end': '2026-02-12 22:26:45'},
    {'label': 'NOAA',      'cert_key': 'NOAA',
     'stable_start': '2026-02-12 22:46:00', 'stable_end': '2026-02-12 22:52:00'},
    {'label': 'Dilution1', 'cert_key': 'Dilution1',
     'stable_start': '2026-02-12 22:57:00', 'stable_end': '2026-02-12 22:58:00'},
    {'label': 'Dilution2', 'cert_key': 'Dilution2',
     'stable_start': '2026-02-12 23:01:00', 'stable_end': '2026-02-12 23:03:00'},
    {'label': 'Dilution3', 'cert_key': 'Dilution3',
     'stable_start': '2026-02-12 23:03:30', 'stable_end': '2026-02-12 23:04:30'},
    {'label': 'Dilution4', 'cert_key': 'Dilution4',
     'stable_start': '2026-02-12 23:05:30', 'stable_end': '2026-02-12 23:07:00'},
    {'label': 'Dilution5', 'cert_key': 'Dilution5',
     'stable_start': '2026-02-12 23:08:00', 'stable_end': '2026-02-12 23:09:00'},
]

FEB6_WINDOWS = [
    {'label': 'N2_zero',      'cert_key': 'N2_zero',
     'stable_start': '2026-02-06 18:15:45', 'stable_end': '2026-02-06 18:16:20'},
    {'label': 'Dilution3',    'cert_key': 'Dilution3',
     'stable_start': '2026-02-06 18:18:00', 'stable_end': '2026-02-06 18:19:20'},
    {'label': 'Dilution2',    'cert_key': 'Dilution2',
     'stable_start': '2026-02-06 18:20:00', 'stable_end': '2026-02-06 18:21:30'},
    {'label': 'Dilution1',    'cert_key': 'Dilution1',
     'stable_start': '2026-02-06 18:22:20', 'stable_end': '2026-02-06 18:23:40'},
    {'label': 'Dilution4',    'cert_key': 'Dilution4',
     'stable_start': '2026-02-06 18:27:00', 'stable_end': '2026-02-06 18:28:30'},
    {'label': 'N2_zero (end)', 'cert_key': 'N2_zero',
     'stable_start': '2026-02-06 18:29:30', 'stable_end': '2026-02-06 18:30:30'},
]

In [ ]:
CH4_COLS  = ['CH4_ppm_Picarro', 'CH4_ppm_Ultra460', 'CH4_ppm_Ultra321', 'CH4_ppm_Pico017']
C3H8_COLS = ['C3H8_ppm_Ultra321']
C2H6_COLS = ['C2H6_ppb_Ultra460', 'C2H6_ppb_Pico017', 'C2H6_ppm_Ultra321']


def load_merged(date_str):
    df = pd.read_csv(MERGED_DIR / f'{date_str}.csv')
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
    return df


def extract_medians(df, windows):
    """Return a DataFrame with per-window median for each instrument column."""
    all_cols = [c for c in CH4_COLS + C3H8_COLS + C2H6_COLS if c in df.columns]
    rows = []
    for w in windows:
        t0 = pd.Timestamp(w['stable_start'])
        t1 = pd.Timestamp(w['stable_end'])
        sub = df[(df['TIMESTAMP'] >= t0) & (df['TIMESTAMP'] <= t1)]
        row = {'label': w['label'], 'cert_key': w['cert_key'], 'n_rows': len(sub)}
        for col in all_cols:
            row[col] = sub[col].median()
        rows.append(row)
    return pd.DataFrame(rows)


def linreg(x, y):
    """Linear regression dropping NaN pairs.  Returns (slope, intercept, r2)."""
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan, np.nan, np.nan
    sl, ic, r, *_ = stats.linregress(x[mask], y[mask])
    return sl, ic, r**2


# Window shade colours (cert_key → colour)
W_COLORS = {
    'N2_zero':   '#a8d8ea',
    'NOAA':      '#b8f0b8',
    'Dilution1': '#fffacd',
    'Dilution2': '#ffd700',
    'Dilution3': '#ffa040',
    'Dilution4': '#ff6030',
    'Dilution5': '#cc44cc',
}


def plot_cal_overview(df, windows, title):
    """CH4 and C3H8 time series with shaded stable calibration windows."""
    t0 = pd.Timestamp(windows[0]['stable_start']) - pd.Timedelta('10min')
    t1 = pd.Timestamp(windows[-1]['stable_end'])  + pd.Timedelta('10min')
    sub = df[(df['TIMESTAMP'] >= t0) & (df['TIMESTAMP'] <= t1)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(title, fontsize=12, fontweight='bold')

    for inst, col in [('Picarro', 'CH4_ppm_Picarro'), ('Ultra460', 'CH4_ppm_Ultra460'),
                      ('Ultra321', 'CH4_ppm_Ultra321'), ('Pico017',  'CH4_ppm_Pico017')]:
        if col in sub.columns:
            ax1.plot(sub['TIMESTAMP'], sub[col], lw=0.6, alpha=0.85,
                     color=COLORS[inst], label=inst)
    ax1.set_ylabel('CH4 (ppm)')
    ax1.legend(loc='upper left', fontsize=8, ncol=2)

    if 'C3H8_ppm_Ultra321' in sub.columns:
        ax2.plot(sub['TIMESTAMP'], sub['C3H8_ppm_Ultra321'],
                 lw=0.6, color=COLORS['Ultra321'], label='Ultra321')
    ax2.set_ylabel('C3H8 (ppm)')
    ax2.legend(loc='upper left', fontsize=8)

    ylim1 = ax1.get_ylim()
    for w in windows:
        ws = pd.Timestamp(w['stable_start'])
        we = pd.Timestamp(w['stable_end'])
        c  = W_COLORS.get(w['cert_key'], '#dddddd')
        ax1.axvspan(ws, we, alpha=0.35, color=c, zorder=0)
        ax2.axvspan(ws, we, alpha=0.35, color=c, zorder=0)
        ax1.text(ws + (we - ws) / 2, ylim1[1] * 0.97,
                 w['label'], fontsize=7, ha='center', va='top', rotation=60)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout()
    plt.show()


## 1 — Feb 12 Calibration Sequence

Verify that the shaded stable windows fall on the right concentration plateaus before extracting medians.

In [ ]:
df12 = load_merged('20260212')
plot_cal_overview(df12, FEB12_WINDOWS, 'Feb 12 calibration sequence (UTC)')


## 2 — CH4 Calibration

Extract per-window medians, verify Picarro agrees with certified values, then regress each Aeris instrument against Picarro.

> **Model:** `measured = slope × Picarro + intercept`
> **Corrected:** `CH4_corrected = (measured − intercept) / slope`


In [ ]:
med12 = extract_medians(df12, FEB12_WINDOWS)

# Attach certified CH4 and C3H8
med12['cert_CH4_ppm']  = med12['cert_key'].map(lambda k: CERT[k]['CH4_ppm'])
med12['cert_C3H8_ppm'] = med12['cert_key'].map(lambda k: CERT[k]['C3H8_ppm'])

print(med12[['label', 'n_rows', 'cert_CH4_ppm',
             'CH4_ppm_Picarro', 'CH4_ppm_Ultra460',
             'CH4_ppm_Ultra321', 'CH4_ppm_Pico017']].to_string(index=False, float_format='%.4f'))


### 2a — Picarro vs certified (sanity check)

In [ ]:
x = med12['cert_CH4_ppm'].values.astype(float)
y = med12['CH4_ppm_Picarro'].values.astype(float)
sl, ic, r2 = linreg(x, y)
print(f'Picarro = {sl:.4f} × certified + {ic:.4f}   R² = {r2:.5f}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, zorder=3)
for _, row in med12.iterrows():
    ax.annotate(row['label'], (row['cert_CH4_ppm'], row['CH4_ppm_Picarro']),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
xfit = np.linspace(0, x.max() * 1.05, 100)
ax.plot(xfit, sl * xfit + ic, 'k--', lw=1, label=f'slope={sl:.4f}, int={ic:.4f}, R²={r2:.4f}')
ax.plot(xfit, xfit, 'gray', lw=0.8, ls=':', label='1:1')
ax.set_xlabel('Certified CH4 (ppm)')
ax.set_ylabel('Picarro CH4 (ppm)')
ax.set_title('Picarro vs certified')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 2b — Aeris instruments vs Picarro

In [ ]:
aeris_ch4 = [
    ('Ultra460', 'CH4_ppm_Ultra460'),
    ('Ultra321', 'CH4_ppm_Ultra321'),
    ('Pico017',  'CH4_ppm_Pico017'),
]

ch4_coefs_high = {}  # full-range fit (all 7 points, 0–57 ppm)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('CH4 high-range calibration — Aeris vs Picarro (Feb 12, all 7 points)', fontsize=12)

for ax, (inst, col) in zip(axes, aeris_ch4):
    x = med12['CH4_ppm_Picarro'].values.astype(float)
    y = med12[col].values.astype(float) if col in med12.columns else np.full_like(x, np.nan)
    sl, ic, r2 = linreg(x, y)
    ch4_coefs_high[inst] = {'slope': sl, 'intercept': ic, 'r2': r2}

    ax.scatter(x, y, color=COLORS[inst], zorder=3)
    for _, row in med12.iterrows():
        if col in row:
            ax.annotate(row['label'], (row['CH4_ppm_Picarro'], row[col]),
                        fontsize=6, xytext=(3, 2), textcoords='offset points')
    xfit = np.linspace(0, np.nanmax(x) * 1.05, 100)
    ax.plot(xfit, sl * xfit + ic, '--', color=COLORS[inst], lw=1.2,
            label=f'slope={sl:.4f}\nint={ic:.4f}\nR²={r2:.4f}')
    ax.plot(xfit, xfit, 'gray', lw=0.8, ls=':', alpha=0.6, label='1:1')
    ax.set_xlabel('Picarro CH4 (ppm)')
    ax.set_ylabel(f'{inst} CH4 (ppm)')
    ax.set_title(inst)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print('\nCH4 high-range calibration coefficients (all 7 points, measured = slope × Picarro + intercept):')
print(f'  {"Instrument":<12} {"slope":>8} {"intercept":>10} {"R²":>8}')
for inst, c in ch4_coefs_high.items():
    print(f'  {inst:<12} {c["slope"]:>8.5f} {c["intercept"]:>10.5f} {c["r2"]:>8.5f}')


### 2c — Low-range CH4 calibration (N2_zero + NOAA + Dil.1, 0–5.7 ppm)

Fit using only the three lowest-concentration calibration points to accurately capture instrument response at near-ambient concentrations.  The high-range (all 7 points) fit above is dominated by the plume-scale dilutions and cannot simultaneously capture the near-zero and near-ambient endpoints.

> **Piecewise threshold:** 6 ppm — below this uses the low-range coefs; at or above uses the high-range coefs.

In [ ]:
CH4_PW_THRESHOLD = 6.0  # ppm — below: low-range coefs; at/above: high-range coefs

LOW_RANGE_LABELS = {'N2_zero', 'NOAA', 'Dilution1'}
med12_low = med12[med12['label'].isin(LOW_RANGE_LABELS)].copy()

ch4_coefs_low = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('CH4 low-range calibration — Aeris vs Picarro (N2_zero + NOAA + Dil.1 only)', fontsize=12)

for ax, (inst, col) in zip(axes, aeris_ch4):
    x_all = med12['CH4_ppm_Picarro'].values.astype(float)
    y_all = med12[col].values.astype(float) if col in med12.columns else np.full_like(x_all, np.nan)
    x_low = med12_low['CH4_ppm_Picarro'].values.astype(float)
    y_low = med12_low[col].values.astype(float) if col in med12_low.columns else np.full_like(x_low, np.nan)
    sl, ic, r2 = linreg(x_low, y_low)
    ch4_coefs_low[inst] = {'slope': sl, 'intercept': ic, 'r2': r2}

    ax.scatter(x_all, y_all, color='lightgray', s=40, zorder=2, label='high-range pts (not used)')
    ax.scatter(x_low, y_low, color=COLORS[inst], s=70, zorder=3, label='low-range pts (used)')
    for _, row in med12.iterrows():
        if col in row:
            ax.annotate(row['label'], (row['CH4_ppm_Picarro'], row[col]),
                        fontsize=6, xytext=(3, 2), textcoords='offset points')
    xfit_lo  = np.linspace(0, 7, 100)
    xfit_all = np.linspace(0, np.nanmax(x_all) * 1.05, 100)
    c_hi = ch4_coefs_high[inst]
    ax.plot(xfit_all, c_hi['slope'] * xfit_all + c_hi['intercept'], ':',
            color='gray', lw=1, label=f'high: sl={c_hi["slope"]:.4f}, int={c_hi["intercept"]:.4f}')
    ax.plot(xfit_lo, sl * xfit_lo + ic, '--', color=COLORS[inst], lw=1.8,
            label=f'low:  sl={sl:.4f}, int={ic:.4f}, R²={r2:.4f}')
    ax.plot(xfit_all, xfit_all, 'gray', lw=0.8, ls=':', alpha=0.4, label='1:1')
    ax.set_xlabel('Picarro CH4 (ppm)')
    ax.set_ylabel(f'{inst} CH4 (ppm)')
    ax.set_title(inst)
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()

print(f'\nCH4 low-range coefficients (N2_zero + NOAA + Dil.1, measured = slope × Picarro + intercept):')
print(f'  {"Instrument":<12} {"slope":>8} {"intercept":>10} {"R²":>8}')
for inst, c in ch4_coefs_low.items():
    print(f'  {inst:<12} {c["slope"]:>8.5f} {c["intercept"]:>10.5f} {c["r2"]:>8.5f}')

print(f'\nCalibrated value comparison at key near-ambient concentrations')
print(f'  (corrected = (raw - intercept) / slope, applied to Picarro reading as stand-in):')
print(f'  {"Instrument":<12}  {"@0 ppm":>10}  {"@2 ppm":>10}  {"@5.7 ppm":>10}')
print(f'  {"":12}  {"hi / lo":>10}  {"hi / lo":>10}  {"hi / lo":>10}')
for inst in ['Ultra460', 'Ultra321', 'Pico017']:
    chi = ch4_coefs_high[inst]; clo = ch4_coefs_low[inst]
    def cal(c, v): return (v - c['intercept']) / c['slope']
    print(f'  {inst:<12}  {cal(chi,0):>4.3f}/{cal(clo,0):>4.3f}  '
          f'{cal(chi,2):>4.3f}/{cal(clo,2):>4.3f}  '
          f'{cal(chi,5.736):>5.3f}/{cal(clo,5.736):>5.3f}')


## 3 — C3H8 Calibration (Ultra321 only)

Ultra321 is the only instrument that reports C3H8.  Regress against certified concentrations (N2 zero + Dilution1–5).  NOAA tank has no certified C3H8 so it is excluded.


In [ ]:
# Filter to rows that have a certified C3H8 value
c3h8_med = med12[med12['cert_C3H8_ppm'].notna()].copy()

x = c3h8_med['cert_C3H8_ppm'].values.astype(float)
y = c3h8_med['C3H8_ppm_Ultra321'].values.astype(float)
sl, ic, r2 = linreg(x, y)
c3h8_coef = {'slope': sl, 'intercept': ic, 'r2': r2}
print(f'Ultra321 C3H8 = {sl:.5f} × certified + {ic:.5f}   R² = {r2:.5f}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, color=COLORS['Ultra321'], zorder=3)
for _, row in c3h8_med.iterrows():
    ax.annotate(row['label'], (row['cert_C3H8_ppm'], row['C3H8_ppm_Ultra321']),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
xfit = np.linspace(0, x.max() * 1.05, 100)
ax.plot(xfit, sl * xfit + ic, '--', color=COLORS['Ultra321'], lw=1.2,
        label=f'slope={sl:.4f}, int={ic:.4f}, R²={r2:.4f}')
ax.plot(xfit, xfit, 'gray', lw=0.8, ls=':', alpha=0.6, label='1:1')
ax.set_xlabel('Certified C3H8 (ppm)')
ax.set_ylabel('Ultra321 C3H8 (ppm)')
ax.set_title('Ultra321 C3H8 vs certified (Feb 12)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4 — Validation on Feb 6

Apply the Feb 12 calibration coefficients to the Feb 6 window medians and check residuals against the certified values.  Feb 6 has Dilution1–4 (no NOAA, no Dilution5).


In [ ]:
df6 = load_merged('20260206')
plot_cal_overview(df6, FEB6_WINDOWS, 'Feb 6 calibration sequence (UTC)')


In [ ]:
med6 = extract_medians(df6, FEB6_WINDOWS)
med6['cert_CH4_ppm']  = med6['cert_key'].map(lambda k: CERT[k]['CH4_ppm'])
med6['cert_C3H8_ppm'] = med6['cert_key'].map(lambda k: CERT[k]['C3H8_ppm'])

# Apply all three CH4 calibration approaches to each window median
for inst, col in [('Ultra460', 'CH4_ppm_Ultra460'),
                  ('Ultra321', 'CH4_ppm_Ultra321'),
                  ('Pico017',  'CH4_ppm_Pico017')]:
    if col not in med6.columns:
        continue
    chi = ch4_coefs_high[inst]
    clo = ch4_coefs_low[inst]
    med6[f'{col}_cal_high'] = (med6[col] - chi['intercept']) / chi['slope']
    med6[f'{col}_cal_low']  = (med6[col] - clo['intercept']) / clo['slope']
    below = med6[col] < CH4_PW_THRESHOLD
    med6[f'{col}_cal_pw'] = np.where(
        below,
        (med6[col] - clo['intercept']) / clo['slope'],
        (med6[col] - chi['intercept']) / chi['slope'],
    )

print('Feb 6 CH4 validation — residuals vs Picarro (cal − Picarro):')
print('  Positive = instrument reads high vs Picarro after calibration\n')
hdr = f'  {"label":<22} {"cert":>6} {"Picarro":>8}  {"raw":>8} {"high":>8} {"low":>8} {"pw":>8}'
for inst, col in [('Ultra460', 'CH4_ppm_Ultra460'),
                  ('Ultra321', 'CH4_ppm_Ultra321'),
                  ('Pico017',  'CH4_ppm_Pico017')]:
    if col not in med6.columns:
        continue
    print(f'  ── {inst} ──')
    print(hdr)
    for _, row in med6.iterrows():
        pic = row['CH4_ppm_Picarro']
        raw = row[col]
        rhi = row.get(f'{col}_cal_high', np.nan) - pic
        rlo = row.get(f'{col}_cal_low',  np.nan) - pic
        rpw = row.get(f'{col}_cal_pw',   np.nan) - pic
        pw_tag = 'L' if raw < CH4_PW_THRESHOLD else 'H'
        print(f'  {row["label"]:<22} {row["cert_CH4_ppm"]:>6.3f} {pic:>8.4f}  '
              f'{raw-pic:>+8.4f} {rhi:>+8.4f} {rlo:>+8.4f} {rpw:>+8.4f} ({pw_tag})')
    print()


In [ ]:
# Plot: raw + all three calibrated approaches vs Picarro on Feb 6
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Feb 6 validation — raw vs three CH4 calibration approaches', fontsize=12)

markers = {'raw': ('o', 'lightgray', 30), 'high': ('s', None, 50),
           'low': ('^', None, 50), 'pw': ('D', None, 50)}

for ax, (inst, col) in zip(axes, [('Ultra460', 'CH4_ppm_Ultra460'),
                                    ('Ultra321', 'CH4_ppm_Ultra321'),
                                    ('Pico017',  'CH4_ppm_Pico017')]):
    x = med6['CH4_ppm_Picarro'].values.astype(float)

    ax.scatter(x, med6[col].values.astype(float),
               color='lightgray', s=30, zorder=2, label='raw')
    for tag, lbl in [('cal_high', 'high'), ('cal_low', 'low'), ('cal_pw', 'pw')]:
        key = f'{col}_{tag}'
        if key in med6.columns:
            ax.scatter(x, med6[key].values.astype(float),
                       color=COLORS[inst], s=50, zorder=3,
                       alpha=[0.4, 0.7, 1.0][['cal_high','cal_low','cal_pw'].index(tag)],
                       marker=['^','s','D'][['cal_high','cal_low','cal_pw'].index(tag)],
                       label=lbl)
    for _, row in med6.iterrows():
        if col in row:
            ax.annotate(row['label'], (row['CH4_ppm_Picarro'], row[col]),
                        fontsize=6, xytext=(3, 2), textcoords='offset points', color='gray')

    xfit = np.linspace(0, np.nanmax(x) * 1.05, 100)
    ax.plot(xfit, xfit, 'k:', lw=0.8, label='1:1')
    ax.set_xlabel('Picarro CH4 (ppm)')
    ax.set_ylabel(f'{inst} CH4 (ppm)')
    ax.set_title(inst)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 5 — C2H6 Cross-Calibration (Ultra460 as reference)

The NOAA tank C2H6 (1.63 ppb) is too low to span the ambient range, so we cross-calibrate using all WYO co-deployment days (Feb 5–12).  Ultra460 is taken as truth.  We filter to rows where Ultra460 C2H6 > 20 ppb to ensure meaningful signal above background noise.

- **Pico017** (ppb) calibrated to Ultra460 (ppb)
- **Ultra321** C2H6 is in ppm; we convert to ppb for comparison.  Ultra321 can only detect C2H6 during elevated plume events — background values are below its noise floor.


In [ ]:
WYO_DATES = ['20260205', '20260206', '20260208', '20260209',
             '20260210', '20260211', '20260212']

# Calibration windows to exclude (we don't want tank gas in the cross-cal)
CAL_EXCLUDE = (
    [('2026-02-06 17:25:00', '2026-02-06 18:32:00')]  # Feb 6 cal sequence
  + [('2026-02-12 22:20:00', '2026-02-12 23:15:00')]  # Feb 12 cal sequence
)

dfs = []
for d in WYO_DATES:
    df = load_merged(d)
    # Exclude calibration windows
    mask = pd.Series(True, index=df.index)
    for t0s, t1s in CAL_EXCLUDE:
        t0, t1 = pd.Timestamp(t0s), pd.Timestamp(t1s)
        mask &= ~((df['TIMESTAMP'] >= t0) & (df['TIMESTAMP'] <= t1))
    dfs.append(df[mask])

all_wyo = pd.concat(dfs, ignore_index=True)

# Filter to rows with valid C2H6 on all three instruments and U460 > 20 ppb
c2h6_mask = (
    all_wyo['C2H6_ppb_Ultra460'].notna() &
    all_wyo['C2H6_ppb_Pico017'].notna()  &
    all_wyo['C2H6_ppm_Ultra321'].notna() &
    (all_wyo['C2H6_ppb_Ultra460'] > 20)
)
c2h6_df = all_wyo[c2h6_mask].copy()
# Ultra321 is in ppm — convert to ppb for regression
c2h6_df['C2H6_ppb_Ultra321'] = c2h6_df['C2H6_ppm_Ultra321'] * 1000.0

print(f'Rows with Ultra460 C2H6 > 20 ppb (excl. cal windows): {len(c2h6_df):,}')
print(f'Ultra460 C2H6 range: {c2h6_df["C2H6_ppb_Ultra460"].min():.1f} – {c2h6_df["C2H6_ppb_Ultra460"].max():.1f} ppb')


In [ ]:
from matplotlib.colors import LogNorm

c2h6_coefs = {}

# ── Plot 1: improved hexbin (viridis + log-scale counts) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('C2H6 cross-calibration vs Ultra460 — all data (U460 > 20 ppb)', fontsize=11)

for ax, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                   ('Ultra321', 'C2H6_ppb_Ultra321')]):
    x = c2h6_df['C2H6_ppb_Ultra460'].values
    y = c2h6_df[col].values
    sl, ic, r2 = linreg(x, y)
    c2h6_coefs[inst] = {'slope': sl, 'intercept': ic, 'r2': r2}

    hb = ax.hexbin(x, y, gridsize=60, cmap='viridis', mincnt=1,
                   norm=LogNorm(), linewidths=0.2)
    plt.colorbar(hb, ax=ax, label='count (log scale)')
    xfit = np.linspace(x.min(), x.max(), 200)
    ax.plot(xfit, sl * xfit + ic, 'r-', lw=1.5,
            label=f'slope={sl:.4f}\nint={ic:.2f} ppb\nR²={r2:.4f}')
    ax.plot(xfit, xfit, 'w:', lw=1.2, label='1:1')
    ax.set_xlabel('Ultra460 C2H6 (ppb)')
    unit = 'ppb (×1000 from ppm)' if inst == 'Ultra321' else 'ppb'
    ax.set_ylabel(f'{inst} C2H6 ({unit})')
    ax.set_title(inst)
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

# ── Plot 2: scatter colored by CH4 (spectral interference check) ──────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('C2H6 scatter colored by CH4 — spectral interference check', fontsize=11)

for ax, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                   ('Ultra321', 'C2H6_ppb_Ultra321')]):
    x   = c2h6_df['C2H6_ppb_Ultra460'].values
    y   = c2h6_df[col].values
    ch4 = c2h6_df['CH4_ppm_Picarro'].values

    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(ch4)
    sc = ax.scatter(x[valid], y[valid], c=ch4[valid], cmap='plasma',
                    norm=LogNorm(vmin=max(ch4[valid].min(), 0.5), vmax=ch4[valid].max()),
                    s=4, alpha=0.5, rasterized=True)
    plt.colorbar(sc, ax=ax, label='CH4 Picarro (ppm, log scale)')
    c = c2h6_coefs[inst]
    xfit = np.linspace(x[valid].min(), x[valid].max(), 200)
    ax.plot(xfit, c['slope'] * xfit + c['intercept'], 'k-', lw=1.5, label='all-data fit')
    ax.plot(xfit, xfit, 'k:', lw=0.8, alpha=0.7, label='1:1')
    ax.set_xlabel('Ultra460 C2H6 (ppb)')
    unit = 'ppb (converted)' if inst == 'Ultra321' else 'ppb'
    ax.set_ylabel(f'{inst} C2H6 ({unit})')
    ax.set_title(inst)
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

# ── Plot 3: residuals vs CH4 (trend = interference) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('C2H6 residuals vs CH4 — a nonzero trend indicates spectral interference', fontsize=11)

for ax, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                   ('Ultra321', 'C2H6_ppb_Ultra321')]):
    c   = c2h6_coefs[inst]
    x   = c2h6_df['C2H6_ppb_Ultra460'].values
    y   = c2h6_df[col].values
    ch4 = c2h6_df['CH4_ppm_Picarro'].values
    resid = y - (c['slope'] * x + c['intercept'])

    valid = np.isfinite(ch4) & np.isfinite(resid)
    ax.scatter(ch4[valid], resid[valid], s=3, alpha=0.3,
               color=COLORS[inst], rasterized=True)
    ax.axhline(0, color='k', lw=0.8, ls='--')

    # Percentile-bin medians to reveal trend without outlier distortion
    edges = np.percentile(ch4[valid], np.linspace(0, 100, 25))
    idx   = np.digitize(ch4[valid], edges)
    bx = [np.median(ch4[valid][idx == i])   for i in range(1, len(edges)) if (idx == i).sum() > 2]
    by = [np.median(resid[valid][idx == i])  for i in range(1, len(edges)) if (idx == i).sum() > 2]
    ax.plot(bx, by, 'r-o', lw=1.5, ms=4, label='bin median')

    ax.set_xlabel('CH4 Picarro (ppm)')
    ax.set_ylabel(f'Residual: {inst} − predicted (ppb)')
    ax.set_title(inst)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── CH4-filtered regression: avoid plume-period spectral interference ─────
# If the residuals plot shows a rising trend with CH4, this filtered version
# is more trustworthy.  Adjust CH4_FILTER based on what you see above.

CH4_FILTER = 3.0   # ppm

c2h6_filt = c2h6_df[c2h6_df['CH4_ppm_Picarro'].notna() &
                     (c2h6_df['CH4_ppm_Picarro'] < CH4_FILTER)].copy()
print(f'Filtered: U460 C2H6 > 20 ppb AND Picarro CH4 < {CH4_FILTER} ppm → {len(c2h6_filt):,} rows')
print(f'  U460 C2H6 range: {c2h6_filt["C2H6_ppb_Ultra460"].min():.1f}'
      f' – {c2h6_filt["C2H6_ppb_Ultra460"].max():.1f} ppb')

c2h6_coefs_filt = {}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'C2H6: all-data vs CH4-filtered (CH4 < {CH4_FILTER} ppm) regression', fontsize=11)

for ax, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                   ('Ultra321', 'C2H6_ppb_Ultra321')]):
    x_all = c2h6_df['C2H6_ppb_Ultra460'].values
    y_all = c2h6_df[col].values
    x_f   = c2h6_filt['C2H6_ppb_Ultra460'].values
    y_f   = c2h6_filt[col].values
    sl_f, ic_f, r2_f = linreg(x_f, y_f)
    c2h6_coefs_filt[inst] = {'slope': sl_f, 'intercept': ic_f, 'r2': r2_f}
    c_all = c2h6_coefs[inst]

    ax.scatter(x_all, y_all, s=2, alpha=0.15, color='lightgray',
               rasterized=True, label='all data')
    ax.scatter(x_f,   y_f,   s=2, alpha=0.45, color=COLORS[inst],
               rasterized=True, label=f'CH4 < {CH4_FILTER} ppm')

    xfit = np.linspace(0, np.nanmax(x_all) * 1.02, 200)
    ax.plot(xfit, c_all['slope'] * xfit + c_all['intercept'], 'k-', lw=1.2,
            label=f'all:  slope={c_all["slope"]:.3f}, int={c_all["intercept"]:.1f}, R²={c_all["r2"]:.3f}')
    ax.plot(xfit, sl_f * xfit + ic_f, '--', color=COLORS[inst], lw=1.5,
            label=f'filt: slope={sl_f:.3f}, int={ic_f:.1f}, R²={r2_f:.3f}')
    ax.plot(xfit, xfit, 'k:', lw=0.8, alpha=0.5, label='1:1')

    ax.set_xlabel('Ultra460 C2H6 (ppb)')
    unit = 'ppb (converted)' if inst == 'Ultra321' else 'ppb'
    ax.set_ylabel(f'{inst} C2H6 ({unit})')
    ax.set_title(inst)
    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()

print('\nCoefficient comparison (all-data vs CH4-filtered):')
print(f'  {"Instrument":<12}  {"all slope":>10} {"all int":>9} {"all R²":>7}  '
      f'{"filt slope":>10} {"filt int":>9} {"filt R²":>7}')
for inst in ['Pico017', 'Ultra321']:
    ca = c2h6_coefs[inst]
    cf = c2h6_coefs_filt[inst]
    print(f'  {inst:<12}  {ca["slope"]:>10.5f} {ca["intercept"]:>9.3f} {ca["r2"]:>7.4f}  '
          f'{cf["slope"]:>10.5f} {cf["intercept"]:>9.3f} {cf["r2"]:>7.4f}')

# ── Choose which C2H6 coefficients to carry forward ───────────────────────
# Uncomment one:
#c2h6_coefs_final = c2h6_coefs        # all-data
c2h6_coefs_final = c2h6_coefs_filt # CH4-filtered
print(f'\nUsing: {"all-data" if c2h6_coefs_final is c2h6_coefs else "CH4-filtered"} C2H6 coefficients')


In [ ]:
# ── C3H8 / C2H6 cross-interference diagnostic ────────────────────────────────
# OA-ICOS instruments measure near 3.3 μm where C3H8 has overlapping absorption.
# If residuals from the all-data C2H6 fit correlate with C3H8 level, propane
# cross-sensitivity is the dominant source of scatter (not CH4, not timing).

c2h6_df_diag = c2h6_df.copy()
c2h6_df_diag['C3H8_ppb'] = c2h6_df_diag['C3H8_ppm_Ultra321'] * 1000.0
diag = c2h6_df_diag.dropna(subset=['C3H8_ppb'])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('C2H6 cross-interference: scatter colored by C3H8 and residuals vs C3H8', fontsize=12)

for row_axes, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                         ('Ultra321', 'C2H6_ppb_Ultra321')]):
    sub = diag.dropna(subset=[col])
    x    = sub['C2H6_ppb_Ultra460'].values
    y    = sub[col].values
    c3h8 = sub['C3H8_ppb'].values
    c    = c2h6_coefs[inst]
    resid = y - (c['slope'] * x + c['intercept'])

    # Left: scatter colored by C3H8 level
    ax = row_axes[0]
    sc = ax.scatter(x, y, c=c3h8, cmap='plasma', s=3, alpha=0.4,
                    vmin=0, vmax=np.percentile(c3h8[np.isfinite(c3h8)], 98),
                    rasterized=True)
    plt.colorbar(sc, ax=ax, label='C3H8 Ultra321 (ppb)')
    xfit = np.linspace(0, x.max() * 1.02, 200)
    ax.plot(xfit, c['slope'] * xfit + c['intercept'], 'w-', lw=1.5, label='all-data fit')
    ax.plot(xfit, xfit, 'w:', lw=1.0, alpha=0.8, label='1:1')
    ax.set_xlabel('Ultra460 C2H6 (ppb)')
    ax.set_ylabel(f'{inst} C2H6 (ppb)')
    ax.set_title(f'{inst} — colored by C3H8')
    ax.legend(fontsize=7)

    # Right: residuals vs C3H8 with percentile-bin median trend
    ax = row_axes[1]
    valid = np.isfinite(c3h8) & np.isfinite(resid)
    ax.scatter(c3h8[valid], resid[valid], s=3, alpha=0.3,
               color=COLORS[inst], rasterized=True)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    edges  = np.percentile(c3h8[valid], np.linspace(0, 100, 25))
    idx_b  = np.digitize(c3h8[valid], edges)
    bx = [np.median(c3h8[valid][idx_b == k]) for k in range(1, len(edges)) if (idx_b == k).sum() > 2]
    by = [np.median(resid[valid][idx_b == k]) for k in range(1, len(edges)) if (idx_b == k).sum() > 2]
    ax.plot(bx, by, 'r-o', ms=4, lw=1.5, label='bin median')
    ax.set_xlabel('C3H8 Ultra321 (ppb)')
    ax.set_ylabel(f'Residual: {inst} − predicted (ppb)')
    ax.set_title(f'{inst} residuals vs C3H8')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
from scipy.signal import find_peaks as sp_find_peaks

PEAK_THRESHOLD = 50    # ppb — minimum Ultra460 C2H6 height above baseline
PEAK_WINDOW_S  = 10    # ± seconds to collect around each reference peak
MIN_PROMINENCE = 15    # ppb


def find_peak_pairs(df, ref_col='C2H6_ppb_Ultra460',
                    target_cols=None, window_s=PEAK_WINDOW_S):
    """For each plume peak in ref_col, collect each instrument's max within ±window_s s."""
    if target_cols is None:
        target_cols = ['C2H6_ppb_Pico017', 'C2H6_ppm_Ultra321',
                       'CH4_ppm_Picarro', 'C3H8_ppm_Ultra321']
    records = []
    for date, grp in df.groupby(df['TIMESTAMP'].dt.date):
        grp = grp.set_index('TIMESTAMP').sort_index()
        valid = grp[ref_col].dropna()
        if len(valid) < 20:
            continue
        peaks, _ = sp_find_peaks(valid.values,
                                  height=PEAK_THRESHOLD,
                                  distance=30,
                                  prominence=MIN_PROMINENCE)
        for p in peaks:
            t_peak = valid.index[p]
            win = grp[(grp.index >= t_peak - pd.Timedelta(seconds=window_s)) &
                      (grp.index <= t_peak + pd.Timedelta(seconds=window_s))]
            row = {'date': str(date), 'peak_time': t_peak,
                   ref_col: win[ref_col].max()}
            for col in target_cols:
                if col in win.columns:
                    row[col] = win[col].max()
                else:
                    row[col] = np.nan
            records.append(row)
    return pd.DataFrame(records)


peaks_df = find_peak_pairs(all_wyo)
peaks_df['C2H6_ppb_Ultra321'] = peaks_df['C2H6_ppm_Ultra321'] * 1000.0
print(f'Plume peaks found: {len(peaks_df)}')
print(peaks_df.groupby('date').size().rename('n_peaks').to_string())


In [ ]:
# ── Peak-alignment regression ─────────────────────────────────────────────
# Compare peaks found above — each data point is one plume event, not one second.
# Points are colored by deployment day so per-day drift is visible.

c2h6_coefs_peaks = {}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('C2H6 peak-alignment calibration vs Ultra460', fontsize=11)

for ax, (inst, col) in zip(axes, [('Pico017',  'C2H6_ppb_Pico017'),
                                   ('Ultra321', 'C2H6_ppb_Ultra321')]):
    sub = peaks_df.dropna(subset=['C2H6_ppb_Ultra460', col])
    x = sub['C2H6_ppb_Ultra460'].values
    y = sub[col].values
    sl, ic, r2 = linreg(x, y)
    c2h6_coefs_peaks[inst] = {'slope': sl, 'intercept': ic, 'r2': r2}
    c_all = c2h6_coefs[inst]

    unique_dates = sorted(sub['date'].unique())
    palette = plt.cm.tab10(np.linspace(0, 1, len(unique_dates)))
    for i, d in enumerate(unique_dates):
        mask = (sub['date'] == d).values
        ax.scatter(x[mask], y[mask], color=palette[i], s=40,
                   zorder=3, label=d, alpha=0.85)

    xfit = np.linspace(0, np.nanmax(x) * 1.05, 200)
    ax.plot(xfit, c_all['slope'] * xfit + c_all['intercept'], 'k--', lw=1,
            label=f'all-data: R²={c_all["r2"]:.3f}')
    ax.plot(xfit, sl * xfit + ic, '-', color=COLORS[inst], lw=2,
            label=f'peaks: slope={sl:.3f}, int={ic:.1f}, R²={r2:.3f}')
    ax.plot(xfit, xfit, 'k:', lw=0.8, alpha=0.5, label='1:1')
    ax.set_xlabel('Ultra460 C2H6 peak (ppb)')
    unit = 'ppb (converted)' if inst == 'Ultra321' else 'ppb'
    ax.set_ylabel(f'{inst} C2H6 peak ({unit})')
    ax.set_title(inst)
    ax.legend(fontsize=6, loc='upper left', ncol=2)

plt.tight_layout()
plt.show()

print('\nPeak-alignment vs all-data regression:')
print(f'  {"Instrument":<12}  {"all slope":>10} {"all int":>8} {"all R²":>7}  '
      f'{"peak slope":>10} {"peak int":>8} {"peak R²":>7}')
for inst in ['Pico017', 'Ultra321']:
    ca = c2h6_coefs[inst]
    cp = c2h6_coefs_peaks[inst]
    print(f'  {inst:<12}  {ca["slope"]:>10.5f} {ca["intercept"]:>8.3f} {ca["r2"]:>7.4f}  '
          f'{cp["slope"]:>10.5f} {cp["intercept"]:>8.3f} {cp["r2"]:>7.4f}')

# ── Final C2H6 coefficient selection ──────────────────────────────────────
# Uncomment the set you want carried into the summary:
# c2h6_coefs_final = c2h6_coefs        # all-data
# c2h6_coefs_final = c2h6_coefs_filt   # CH4-filtered
c2h6_coefs_final = c2h6_coefs_peaks    # peak-alignment (recommended)
print(f'\nUsing: peak-alignment C2H6 coefficients.')


## 6 — Summary of Calibration Coefficients

In [ ]:
print('=' * 70)
print('CALIBRATION SUMMARY')
print('=' * 70)
print()
print('CH4  (corrected = (measured - intercept) / slope)')
print(f'  Calibration date : Feb 12 2026 (UTC)')
print(f'  Reference        : Picarro G2401')
print(f'  Piecewise threshold : {CH4_PW_THRESHOLD} ppm  (low-range below, high-range at/above)')
print()
print(f'  High-range (all 7 points, 0–57 ppm):')
print(f'  {"Instrument":<12} {"slope":>9} {"intercept":>11} {"R²":>8}')
for inst, c in ch4_coefs_high.items():
    print(f'  {inst:<12} {c["slope"]:>9.5f} {c["intercept"]:>11.5f} {c["r2"]:>8.5f}')
print()
print(f'  Low-range (N2_zero + NOAA + Dil.1, 0–5.7 ppm):')
print(f'  {"Instrument":<12} {"slope":>9} {"intercept":>11} {"R²":>8}')
for inst, c in ch4_coefs_low.items():
    print(f'  {inst:<12} {c["slope"]:>9.5f} {c["intercept"]:>11.5f} {c["r2"]:>8.5f}')
print()
print('C3H8  (corrected = (measured - intercept) / slope)')
print(f'  Instrument       : Ultra321 only')
print(f'  Reference        : certified tank concentrations')
print(f'  {"slope":>9} {"intercept":>11} {"R²":>8}')
print(f'  {c3h8_coef["slope"]:>9.5f} {c3h8_coef["intercept"]:>11.5f} {c3h8_coef["r2"]:>8.5f}')
print()
if c2h6_coefs_final is c2h6_coefs:
    c2h6_label = 'all-data'
elif c2h6_coefs_final is c2h6_coefs_filt:
    c2h6_label = f'CH4-filtered (<{CH4_FILTER} ppm)'
else:
    c2h6_label = 'peak-alignment'
print(f'C2H6  (corrected = (measured - intercept) / slope)  [{c2h6_label}]')
print(f'  Reference        : Ultra460')
print(f'  Data             : all WYO days Feb 5–12, Ultra460 > 20 ppb')
print(f'  Note             : Ultra321 converted ppm → ppb before regression')
print()
print(f'  {"Instrument":<12} {"slope":>9} {"intercept":>11} {"R²":>8}')
for inst, c in c2h6_coefs_final.items():
    print(f'  {inst:<12} {c["slope"]:>9.5f} {c["intercept"]:>11.4f} {c["r2"]:>8.5f}')
print('=' * 70)


In [ ]:
import json as _json
from datetime import datetime as _dt

COEFS_OUT = Path('/uufs/chpc.utah.edu/common/home/u0890904/LAIR_1/Projects/Mobile_SLV'
                 '/Code/mobile-slv/offsets/calibration_coefs.json')

corrections = []

# CH4 — three calibration columns per instrument:
#   _cal_high : full-range OLS (all 7 points, 0–57 ppm)
#   _cal_low  : low-range OLS (N2_zero + NOAA + Dil.1, 0–5.7 ppm)
#   _cal_pw   : piecewise (low below CH4_PW_THRESHOLD, high at/above)
for inst, col in [('Ultra460', 'CH4_ppm_Ultra460'),
                  ('Ultra321', 'CH4_ppm_Ultra321'),
                  ('Pico017',  'CH4_ppm_Pico017')]:
    chi = ch4_coefs_high[inst]
    clo = ch4_coefs_low[inst]
    corrections.append({
        'gas': 'CH4', 'instrument': inst, 'type': 'linear',
        'col_in': col, 'col_out': col + '_cal_high',
        'scale_in': 1.0,
        'slope':     round(chi['slope'],     8),
        'intercept': round(chi['intercept'], 8),
        'r2':        round(chi['r2'],        6),
    })
    corrections.append({
        'gas': 'CH4', 'instrument': inst, 'type': 'linear',
        'col_in': col, 'col_out': col + '_cal_low',
        'scale_in': 1.0,
        'slope':     round(clo['slope'],     8),
        'intercept': round(clo['intercept'], 8),
        'r2':        round(clo['r2'],        6),
    })
    corrections.append({
        'gas': 'CH4', 'instrument': inst, 'type': 'piecewise_linear',
        'col_in': col, 'col_out': col + '_cal_pw',
        'threshold_ppm': CH4_PW_THRESHOLD,
        'low':  {'slope': round(clo['slope'], 8), 'intercept': round(clo['intercept'], 8), 'r2': round(clo['r2'], 6)},
        'high': {'slope': round(chi['slope'], 8), 'intercept': round(chi['intercept'], 8), 'r2': round(chi['r2'], 6)},
    })

# C3H8 — Ultra321 calibrated against certified tank concentrations
corrections.append({
    'gas': 'C3H8', 'instrument': 'Ultra321', 'type': 'linear',
    'col_in': 'C3H8_ppm_Ultra321', 'col_out': 'C3H8_ppm_Ultra321_cal',
    'scale_in': 1.0,
    'slope':     round(c3h8_coef['slope'],     8),
    'intercept': round(c3h8_coef['intercept'], 8),
    'r2':        round(c3h8_coef['r2'],        6),
})

# C2H6 Pico017 — calibrated against Ultra460 (peak-alignment, units: ppb)
c = c2h6_coefs_final['Pico017']
corrections.append({
    'gas': 'C2H6', 'instrument': 'Pico017', 'type': 'linear',
    'col_in': 'C2H6_ppb_Pico017', 'col_out': 'C2H6_ppb_Pico017_cal',
    'scale_in': 1.0,
    'slope':     round(c['slope'],     8),
    'intercept': round(c['intercept'], 6),
    'r2':        round(c['r2'],        6),
})

# C2H6 Ultra321 — merged file stores ppm; calibration was in ppb → scale_in=1000
c = c2h6_coefs_final['Ultra321']
corrections.append({
    'gas': 'C2H6', 'instrument': 'Ultra321', 'type': 'linear',
    'col_in': 'C2H6_ppm_Ultra321', 'col_out': 'C2H6_ppb_Ultra321_cal',
    'scale_in': 1000.0,
    'slope':     round(c['slope'],     8),
    'intercept': round(c['intercept'], 6),
    'r2':        round(c['r2'],        6),
})

payload = {
    'metadata': {
        'calibration_date': '2026-02-12',
        'generated_utc': _dt.utcnow().strftime('%Y-%m-%dT%H:%M'),
        'ch4_pw_threshold_ppm': CH4_PW_THRESHOLD,
        'c2h6_method': 'peak-alignment (scipy.signal.find_peaks on Ultra460, ±10 s window)',
        'formula': ('linear: cal=(col_in*scale_in - intercept)/slope; '
                    'piecewise_linear: low formula if col_in<threshold_ppm, else high formula'),
    },
    'corrections': corrections,
}

COEFS_OUT.write_text(_json.dumps(payload, indent=2))
print(f'Wrote {COEFS_OUT}\n')
print(f'  {"gas":<6} {"instrument":<12} {"type":<18} {"col_in":<26} → col_out')
for corr in corrections:
    ctype = corr.get('type', 'linear')
    print(f'  {corr["gas"]:<6} {corr["instrument"]:<12} {ctype:<18} {corr["col_in"]:<26} → {corr["col_out"]}')
